# Dataset Exploration & Audio Analysis

Comprehensive exploration of **EARS** (Expressive, Anechoic Recordings of Speech) and **WHAM!** (Noise) datasets used for speech enhancement training.

**Contents:**
1. Dataset Loading & Metadata Overview
2. EARS Dataset Metadata Analysis
3. WHAM! Noise Dataset Metadata Analysis
4. Audio Waveform Exploration (torchaudio)
5. Spectrogram Analysis & Comparisons
6. Clean vs. Noisy Signal Mixing & Visualization
7. Data Pipeline Visualization

In [ ]:
import os
import sys
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import torchaudio
import torchaudio.transforms as T
import torchaudio.functional as F
import IPython.display as ipd

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.metadata import (
    get_ears_personal_metadata,
    preprocess_ears_metadata,
    preprocess_wham_metadata,
    merge_ears_filepaths_with_metadata,
    prepare_for_training
)
from src.mel_regan.utils import create_configs, create_dataframes, split_dataframes
from src.configs import AudioPreprocessorConfig
from src.datasets import (
    MelSpectrogramProcessor,
    AmplitudeSpectrogramProcessor,
    AudioToDBScaler,
    MinMaxFixedNormalizer,
    AudioMixingDataset
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

# Paths
COMMON_PATH = "/Users/hubertmaka/Desktop/Studia/MGR"
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset root: {COMMON_PATH}")

---
## 1. Dataset Loading & Metadata Overview

Load EARS and WHAM datasets using the `mel_bin2bin` utility functions.

In [ ]:
# Load dataframes using project utilities
dataframes = create_dataframes(COMMON_PATH, kaggle=False)
ears_df = dataframes["ears_df"]
wham_df = dataframes["wham_df"]

print(f"EARS dataset: {len(ears_df)} samples")
print(f"WHAM dataset: {len(wham_df)} samples")
print(f"\nEARS columns: {list(ears_df.columns)}")
print(f"WHAM columns:  {list(wham_df.columns)}")

In [ ]:
# Quick peek at EARS data
print("=" * 60)
print("EARS Dataset - First Rows")
print("=" * 60)
ears_df.head(10)

In [ ]:
# Quick peek at WHAM data
print("=" * 60)
print("WHAM Dataset - First Rows")
print("=" * 60)
wham_df.head(10)

In [ ]:
# Detailed summary statistics for both datasets
print("=" * 70)
print("EARS DataFrame Summary")
print("=" * 70)
print(f"Shape: {ears_df.shape}")
print(f"\nColumn types:")
for col in ears_df.columns:
    print(f"  {col:25s} | dtype={str(ears_df[col].dtype):10s} | unique={ears_df[col].nunique()}")
print(f"\nUnique persons:  {ears_df['person'].nunique()}")
print(f"Unique styles:   {sorted(ears_df['style'].unique())}")
print(f"Unique emotions: {sorted(ears_df['emotion'].unique())}")
print(f"Freeform speech: {ears_df['is_freeform_speech'].sum()} ({ears_df['is_freeform_speech'].mean()*100:.1f}%)")
print(f"Non-speech:      {ears_df['is_non_speech'].sum()} ({ears_df['is_non_speech'].mean()*100:.1f}%)")

print(f"\n{'=' * 70}")
print("WHAM DataFrame Summary")
print("=" * 70)
print(f"Shape: {wham_df.shape}")
print(f"\nColumn types:")
for col in wham_df.columns:
    print(f"  {col:35s} | dtype={str(wham_df[col].dtype):10s} | unique={wham_df[col].nunique()}")
print(f"\nNoise Bands:           {sorted(wham_df['Noise Band'].unique())}")
print(f"Reverberation Levels:  {sorted(wham_df['Reverberation Level'].unique())} (h=high, m=medium, l=low)")
print(f"Mic Widths:            {sorted(wham_df['L to R Width (cm)'].unique())}")
print(f"Recording Locations:   {wham_df['Location ID'].nunique()}")
print(f"Noise File IDs:        {wham_df['File ID'].nunique()}")
print(f"SNR range:             [{wham_df['target_speaker1_snr_db'].min():.2f}, {wham_df['target_speaker1_snr_db'].max():.2f}] dB")

---
## 2. EARS Dataset Metadata Analysis

Analyze speaker demographics, speaking styles, emotions, and content distribution.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# 2.1 Speaking Style Distribution
style_counts = ears_df["style"].value_counts()
colors_style = sns.color_palette("viridis", len(style_counts))
bars = axes[0, 0].bar(style_counts.index, style_counts.values, color=colors_style, edgecolor="black", linewidth=0.5)
axes[0, 0].set_title("Speaking Style Distribution", fontsize=14, fontweight="bold")
axes[0, 0].set_xlabel("Style")
axes[0, 0].set_ylabel("Number of Samples")
axes[0, 0].tick_params(axis="x", rotation=45)
for bar, val in zip(bars, style_counts.values):
    axes[0, 0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10, str(val),
                    ha="center", va="bottom", fontsize=9)

# 2.2 Emotion Distribution
emotion_counts = ears_df["emotion"].value_counts()
top_emotions = emotion_counts.head(15)
colors_emo = sns.color_palette("magma", len(top_emotions))
axes[0, 1].barh(top_emotions.index, top_emotions.values, color=colors_emo, edgecolor="black", linewidth=0.5)
axes[0, 1].set_title("Top 15 Emotion Distribution", fontsize=14, fontweight="bold")
axes[0, 1].set_xlabel("Number of Samples")
axes[0, 1].invert_yaxis()
for i, val in enumerate(top_emotions.values):
    axes[0, 1].text(val + 5, i, str(val), va="center", fontsize=9)

# 2.3 Samples per Person
person_counts = ears_df["person"].value_counts().sort_index()
axes[1, 0].bar(range(len(person_counts)), person_counts.values, color="steelblue", alpha=0.8)
axes[1, 0].set_title("Samples per Person", fontsize=14, fontweight="bold")
axes[1, 0].set_xlabel("Person Index")
axes[1, 0].set_ylabel("Number of Samples")
axes[1, 0].axhline(y=person_counts.mean(), color="red", linestyle="--", label=f"Mean: {person_counts.mean():.1f}")
axes[1, 0].legend()

# 2.4 Content Type Distribution (Speech vs Non-Speech vs Freeform)
content_data = {
    "Freeform Speech": ears_df["is_freeform_speech"].sum(),
    "Non-Speech": ears_df["is_non_speech"].sum(),
    "Standard Speech": len(ears_df) - ears_df["is_freeform_speech"].sum() - ears_df["is_non_speech"].sum()
}
wedges, texts, autotexts = axes[1, 1].pie(
    content_data.values(), labels=content_data.keys(),
    autopct="%1.1f%%", colors=sns.color_palette("Set2", 3),
    startangle=140, pctdistance=0.85,
    wedgeprops={"edgecolor": "black", "linewidth": 0.5}
)
axes[1, 1].set_title("Content Type Distribution", fontsize=14, fontweight="bold")

plt.suptitle("EARS Dataset - Metadata Overview", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Speaker demographics from personal metadata
# Columns: age (ranges), ethnicity, gender, weight, native language, height
personal_meta = get_ears_personal_metadata(
    os.path.join(COMMON_PATH, "ears_dataset", "speaker_statistics.json")
)

fig, axes = plt.subplots(2, 3, figsize=(22, 12))

# 1. Gender Distribution
gender_counts = personal_meta["gender"].value_counts()
colors_gender = ["#5dade2", "#f1948a", "#abebc6", "#d7bde2"]
axes[0, 0].bar(gender_counts.index, gender_counts.values,
            color=colors_gender[:len(gender_counts)], edgecolor="black", linewidth=0.5)
axes[0, 0].set_title("Gender Distribution", fontweight="bold", fontsize=13)
axes[0, 0].set_ylabel("Number of Speakers")
for i, val in enumerate(gender_counts.values):
    axes[0, 0].text(i, val + 0.5, str(val), ha="center", fontweight="bold", fontsize=11)
axes[0, 0].tick_params(axis="x", rotation=15)

# 2. Age Distribution (categorical ranges: 18-25, 26-35, 36-45, 46-55, 56-65, 66-75)
age_order = ["18-25", "26-35", "36-45", "46-55", "56-65", "66-75", "prefer not to answer"]
age_counts = personal_meta["age"].value_counts().reindex(age_order).dropna().astype(int)
colors_age = sns.color_palette("YlOrRd", len(age_counts))
bars = axes[0, 1].bar(age_counts.index, age_counts.values, color=colors_age, edgecolor="black", linewidth=0.5)
axes[0, 1].set_title("Age Group Distribution", fontweight="bold", fontsize=13)
axes[0, 1].set_xlabel("Age Range")
axes[0, 1].set_ylabel("Number of Speakers")
axes[0, 1].tick_params(axis="x", rotation=30)
for bar, val in zip(bars, age_counts.values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val), ha="center", fontsize=10)

# 3. Ethnicity Distribution
ethnicity_counts = personal_meta["ethnicity"].value_counts()
colors_eth = sns.color_palette("Set2", len(ethnicity_counts))
wedges, texts, autotexts = axes[0, 2].pie(
    ethnicity_counts.values, labels=None,
    autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct/100.*sum(ethnicity_counts.values)))})",
    colors=colors_eth, startangle=140,
    wedgeprops={"edgecolor": "black", "linewidth": 0.5}, pctdistance=0.75
)
axes[0, 2].legend(ethnicity_counts.index, loc="lower left", fontsize=8, title="Ethnicity")
axes[0, 2].set_title("Ethnicity Distribution", fontweight="bold", fontsize=13)

# 4. Native Language Distribution
lang_counts = personal_meta["native language"].value_counts()
colors_lang = sns.color_palette("viridis", len(lang_counts))
axes[1, 0].barh(lang_counts.index, lang_counts.values, color=colors_lang, edgecolor="black", linewidth=0.5)
axes[1, 0].set_title("Native Language", fontweight="bold", fontsize=13)
axes[1, 0].set_xlabel("Number of Speakers")
axes[1, 0].invert_yaxis()
for i, val in enumerate(lang_counts.values):
    axes[1, 0].text(val + 0.3, i, str(val), va="center", fontsize=10)

# 5. Height Distribution
height_order = ["< 5'", "5' - 5'3", "5'4 - 5'7", "5'8 - 5'11", "6' - 6'3", "prefer not to answer"]
height_counts = personal_meta["height"].value_counts().reindex(height_order).dropna().astype(int)
colors_h = sns.color_palette("cool", len(height_counts))
bars = axes[1, 1].bar(height_counts.index, height_counts.values, color=colors_h, edgecolor="black", linewidth=0.5)
axes[1, 1].set_title("Height Distribution", fontweight="bold", fontsize=13)
axes[1, 1].set_xlabel("Height Range")
axes[1, 1].set_ylabel("Number of Speakers")
axes[1, 1].tick_params(axis="x", rotation=25)
for bar, val in zip(bars, height_counts.values):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val), ha="center", fontsize=10)

# 6. Weight Distribution
weight_order = [
    "100 - 120 lbs", "120 - 140 lbs", "140 - 160 lbs", "160 - 180 lbs",
    "180 - 200 lbs", "200 - 220 lbs", "220 - 240 lbs", "260 - 280 lbs",
    "280 - 300 lbs", "340 - 360 lbs", "prefer not to answer"
]
weight_counts = personal_meta["weight"].value_counts().reindex(weight_order).dropna().astype(int)
colors_w = sns.color_palette("flare", len(weight_counts))
bars = axes[1, 2].bar(range(len(weight_counts)), weight_counts.values, color=colors_w, edgecolor="black", linewidth=0.5)
axes[1, 2].set_xticks(range(len(weight_counts)))
axes[1, 2].set_xticklabels(weight_counts.index, rotation=45, ha="right", fontsize=8)
axes[1, 2].set_title("Weight Distribution", fontweight="bold", fontsize=13)
axes[1, 2].set_xlabel("Weight Range")
axes[1, 2].set_ylabel("Number of Speakers")
for bar, val in zip(bars, weight_counts.values):
    axes[1, 2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val), ha="center", fontsize=9)

plt.suptitle("EARS Dataset - Speaker Demographics (107 Speakers)", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"\nTotal speakers: {len(personal_meta)}")
print(f"Columns: {list(personal_meta.columns)}")
personal_meta.head()

In [ ]:
# Style vs Emotion heatmap
# EARS files follow these patterns:
# - emo_<emotion>_<type>.wav (emotional speech: freeform/sentences)
# - rainbow_<num>_<style>.wav (rainbow passage in 7 styles)  
# - sentences_<num>_<style>.wav (sentences in 7 styles)
# - freeform_speech_<num>.wav (free-form long speech)
# - nonverbal_<type>.wav, vegetative_<type>.wav, interjection_<type>.wav, melodic_<type>.wav (non-speech)

cross = pd.crosstab(ears_df["style"], ears_df["emotion"])
cross_filtered = cross.loc[:, (cross.sum(axis=0) > 0)]

fig, axes = plt.subplots(1, 2, figsize=(24, 8))

sns.heatmap(cross_filtered, annot=True, fmt="d", cmap="YlOrRd", ax=axes[0],
            linewidths=0.5, cbar_kws={"label": "Count"})
axes[0].set_title("Style × Emotion Cross-Tab (Counts)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Emotion Category")
axes[0].set_ylabel("Speaking Style")

# Normalized by style (row)
cross_norm = cross_filtered.div(cross_filtered.sum(axis=1), axis=0)
sns.heatmap(cross_norm, annot=True, fmt=".2f", cmap="YlGnBu", ax=axes[1],
            linewidths=0.5, cbar_kws={"label": "Proportion within Style"})
axes[1].set_title("Style × Emotion (Row-Normalized Proportions)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Emotion Category")
axes[1].set_ylabel("Speaking Style")

plt.suptitle("EARS Dataset: Speaking Style vs Emotion Interaction", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# File type breakdown
print("\nEARS file categories:")
print(f"  Emotional speech (emo_*):     {ears_df['file'].str.startswith('emo_').sum()}")
print(f"  Rainbow passages (rainbow_*): {ears_df['file'].str.startswith('rainbow_').sum()}")
print(f"  Sentences (sentences_*):      {ears_df['file'].str.startswith('sentences_').sum()}")
print(f"  Freeform speech:              {ears_df['is_freeform_speech'].sum()}")
print(f"  Non-speech:                   {ears_df['is_non_speech'].sum()}")

---
## 3. WHAM! Noise Dataset Metadata Analysis

Explore noise characteristics, recording locations, and signal properties.

In [ ]:
# WHAM merged metadata columns:
# utterance_id, Noise Band (0-3), File ID, L to R Width (cm) (15-17cm),
# Reverberation Level (h/l/m), Location ID, Location Day ID,
# noise_samples_beginning_16k, noise_sample_end_16k, target_speaker1_snr_db

print("WHAM Metadata Columns & Summary:")
print("=" * 70)
for col in wham_df.columns:
    u = wham_df[col].nunique()
    nulls = wham_df[col].isnull().sum()
    print(f"  {col:35s} | dtype={str(wham_df[col].dtype):10s} | unique={u:6d} | nulls={nulls}")
print(f"\nTotal samples: {len(wham_df)}")
print(f"\nNumeric summary:")
wham_df.describe()

In [ ]:
# WHAM metadata visualizations - tailored to actual columns
fig, axes = plt.subplots(2, 3, figsize=(22, 12))

# 1. Target Speaker SNR Distribution
snr_data = wham_df["target_speaker1_snr_db"].dropna()
axes[0, 0].hist(snr_data, bins=60, color="#3498db", edgecolor="black", alpha=0.8, density=True)
axes[0, 0].axvline(snr_data.mean(), color="red", linestyle="--", linewidth=2,
                    label=f"Mean: {snr_data.mean():.2f} dB")
axes[0, 0].axvline(snr_data.median(), color="orange", linestyle="-.", linewidth=2,
                    label=f"Median: {snr_data.median():.2f} dB")
axes[0, 0].set_title("Target Speaker SNR Distribution", fontweight="bold", fontsize=13)
axes[0, 0].set_xlabel("SNR (dB)")
axes[0, 0].set_ylabel("Density")
axes[0, 0].legend()

# 2. Noise Band Distribution (categorical: 0, 1, 2, 3)
nb_counts = wham_df["Noise Band"].value_counts().sort_index()
colors_nb = ["#2ecc71", "#3498db", "#e74c3c", "#9b59b6"]
bars = axes[0, 1].bar(nb_counts.index.astype(str), nb_counts.values, color=colors_nb, edgecolor="black")
axes[0, 1].set_title("Noise Band Distribution", fontweight="bold", fontsize=13)
axes[0, 1].set_xlabel("Noise Band ID")
axes[0, 1].set_ylabel("Number of Samples")
for bar, val in zip(bars, nb_counts.values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                    f"{val}\n({val/len(wham_df)*100:.1f}%)", ha="center", fontsize=10)

# 3. Reverberation Level Distribution (h=high, m=medium, l=low)
reverb_map = {"h": "High", "m": "Medium", "l": "Low"}
wham_df["Reverb Label"] = wham_df["Reverberation Level"].map(reverb_map)
reverb_counts = wham_df["Reverb Label"].value_counts().reindex(["Low", "Medium", "High"])
colors_rev = ["#85c1e9", "#f9e79f", "#f1948a"]
wedges, texts, autotexts = axes[0, 2].pie(
    reverb_counts.values, labels=reverb_counts.index,
    autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct/100.*sum(reverb_counts.values)))})",
    colors=colors_rev, startangle=90,
    wedgeprops={"edgecolor": "black", "linewidth": 0.5}, textprops={"fontsize": 11}
)
axes[0, 2].set_title("Reverberation Level", fontweight="bold", fontsize=13)

# 4. L to R Width Distribution (15 cm, 16 cm, 17 cm)
width_counts = wham_df["L to R Width (cm)"].value_counts().sort_index()
colors_w = ["#48c9b0", "#f0b27a", "#bb8fce"]
bars = axes[1, 0].bar(width_counts.index, width_counts.values, color=colors_w, edgecolor="black")
axes[1, 0].set_title("L-to-R Microphone Width", fontweight="bold", fontsize=13)
axes[1, 0].set_xlabel("Width")
axes[1, 0].set_ylabel("Number of Samples")
for bar, val in zip(bars, width_counts.values):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                    f"{val}\n({val/len(wham_df)*100:.1f}%)", ha="center", fontsize=10)

# 5. Noise Sample Beginning Position Distribution
begin_data = wham_df["noise_samples_beginning_16k"].dropna()
axes[1, 1].hist(begin_data, bins=50, color="#e67e22", edgecolor="black", alpha=0.8, density=True)
axes[1, 1].axvline(begin_data.mean(), color="darkred", linestyle="--", linewidth=2,
                    label=f"Mean: {begin_data.mean():.0f}")
axes[1, 1].set_title("Noise Segment Start Position", fontweight="bold", fontsize=13)
axes[1, 1].set_xlabel("Sample Index (@ 16 kHz)")
axes[1, 1].set_ylabel("Density")
axes[1, 1].legend()

# 6. Unique Locations and File IDs
loc_counts = wham_df["Location ID"].value_counts().head(15)
colors_loc = sns.color_palette("Spectral", len(loc_counts))
axes[1, 2].barh(loc_counts.index, loc_counts.values, color=colors_loc, edgecolor="black", linewidth=0.5)
axes[1, 2].set_title(f"Top 15 Recording Locations (of {wham_df['Location ID'].nunique()} total)", fontweight="bold", fontsize=12)
axes[1, 2].set_xlabel("Number of Samples")
axes[1, 2].invert_yaxis()
for i, val in enumerate(loc_counts.values):
    axes[1, 2].text(val + 5, i, str(val), va="center", fontsize=9)

plt.suptitle("WHAM! Noise Dataset - Metadata Analysis", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Clean up temp column
wham_df.drop(columns=["Reverb Label"], inplace=True, errors="ignore")

In [ ]:
# WHAM: Joint distributions and cross-analysis
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# 1. SNR distribution per Noise Band
for band in sorted(wham_df["Noise Band"].unique()):
    subset = wham_df[wham_df["Noise Band"] == band]["target_speaker1_snr_db"]
    axes[0].hist(subset, bins=40, alpha=0.5, density=True, label=f"Band {band} (n={len(subset)})")
axes[0].set_title("SNR Distribution by Noise Band", fontweight="bold", fontsize=13)
axes[0].set_xlabel("SNR (dB)")
axes[0].set_ylabel("Density")
axes[0].legend(fontsize=9)

# 2. SNR distribution per Reverberation Level
reverb_labels = {"h": "High", "m": "Medium", "l": "Low"}
reverb_colors = {"l": "#85c1e9", "m": "#f9e79f", "h": "#f1948a"}
for level in ["l", "m", "h"]:
    subset = wham_df[wham_df["Reverberation Level"] == level]["target_speaker1_snr_db"]
    axes[1].hist(subset, bins=40, alpha=0.5, density=True, color=reverb_colors[level],
                 label=f"{reverb_labels[level]} (n={len(subset)})")
axes[1].set_title("SNR Distribution by Reverberation Level", fontweight="bold", fontsize=13)
axes[1].set_xlabel("SNR (dB)")
axes[1].set_ylabel("Density")
axes[1].legend(fontsize=9)

# 3. Noise Band vs Reverberation Level heatmap
cross = pd.crosstab(wham_df["Noise Band"], wham_df["Reverberation Level"])
cross.columns = [reverb_labels.get(c, c) for c in cross.columns]
sns.heatmap(cross, annot=True, fmt="d", cmap="YlOrRd", ax=axes[2],
            linewidths=1, cbar_kws={"label": "Count"})
axes[2].set_title("Noise Band × Reverberation Level", fontweight="bold", fontsize=13)
axes[2].set_xlabel("Reverberation Level")
axes[2].set_ylabel("Noise Band")

plt.suptitle("WHAM! Cross-Feature Analysis", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Correlation matrix for numeric features
numeric_wham = wham_df[["Noise Band", "noise_samples_beginning_16k", "noise_sample_end_16k", "target_speaker1_snr_db"]]
fig, ax = plt.subplots(figsize=(8, 6))
corr = numeric_wham.corr()
labels = ["Noise Band", "Noise Start\n(samples)", "Noise End\n(samples)", "Target SNR\n(dB)"]
sns.heatmap(corr, annot=True, fmt=".3f", cmap="RdBu_r", center=0, ax=ax,
            linewidths=0.5, vmin=-1, vmax=1, xticklabels=labels, yticklabels=labels,
            cbar_kws={"label": "Correlation Coefficient"})
ax.set_title("WHAM Numeric Features - Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. Audio Waveform Exploration (torchaudio)

Load and visualize raw audio waveforms from both datasets, compute basic audio statistics.

In [ ]:
import soundfile as sf

# Select sample audio files
sample_ears_paths = ears_df.sample(3, random_state=42)["path"].tolist()
sample_wham_paths = wham_df.sample(3, random_state=42)["path"].tolist()

def load_audio_info(path: str) -> dict:
    """Load audio and return waveform + metadata."""
    data, sr = sf.read(path, dtype="float32")
    # soundfile returns (samples,) for mono, (samples, channels) for stereo
    if data.ndim == 1:
        waveform = torch.from_numpy(data).unsqueeze(0)  # [1, samples]
    else:
        waveform = torch.from_numpy(data[:, 0]).unsqueeze(0)  # take first channel
    return {
        "waveform": waveform,
        "sample_rate": sr,
        "channels": 1 if data.ndim == 1 else data.shape[1],
        "samples": waveform.shape[1],
        "duration_s": waveform.shape[1] / sr,
        "max_amplitude": waveform.abs().max().item(),
        "rms": waveform.pow(2).mean().sqrt().item(),
        "filename": os.path.basename(path)
    }

ears_audio = [load_audio_info(p) for p in sample_ears_paths]
wham_audio = [load_audio_info(p) for p in sample_wham_paths]

# Print audio info table
info_rows = []
for a, source in [(a, "EARS") for a in ears_audio] + [(a, "WHAM") for a in wham_audio]:
    info_rows.append({
        "File": a["filename"][:40],
        "SR": a["sample_rate"],
        "Ch": a["channels"],
        "Samples": a["samples"],
        "Duration (s)": f"{a['duration_s']:.2f}",
        "Max Amp": f"{a['max_amplitude']:.4f}",
        "RMS": f"{a['rms']:.4f}",
        "Source": source
    })
pd.DataFrame(info_rows)

In [ ]:
# Plot waveforms: EARS (clean speech) vs WHAM (noise)
fig, axes = plt.subplots(3, 2, figsize=(20, 12), sharex=False)

for i in range(3):
    # EARS clean speech
    ea = ears_audio[i]
    wav = ea["waveform"].squeeze().numpy()
    time_axis = np.arange(len(wav)) / ea["sample_rate"]
    axes[i, 0].plot(time_axis, wav, color="#2ecc71", linewidth=0.3, alpha=0.8)
    axes[i, 0].fill_between(time_axis, wav, alpha=0.15, color="#2ecc71")
    axes[i, 0].set_title(f"EARS: {ea['filename'][:35]}  |  {ea['duration_s']:.1f}s", fontsize=11, fontweight="bold")
    axes[i, 0].set_ylabel("Amplitude")
    axes[i, 0].set_ylim(-1, 1)
    axes[i, 0].axhline(0, color="gray", linewidth=0.5)

    # WHAM noise
    wa = wham_audio[i]
    wav_n = wa["waveform"].squeeze().numpy()
    time_axis_n = np.arange(len(wav_n)) / wa["sample_rate"]
    axes[i, 1].plot(time_axis_n, wav_n, color="#e74c3c", linewidth=0.3, alpha=0.8)
    axes[i, 1].fill_between(time_axis_n, wav_n, alpha=0.15, color="#e74c3c")
    axes[i, 1].set_title(f"WHAM: {wa['filename'][:35]}  |  {wa['duration_s']:.1f}s", fontsize=11, fontweight="bold")
    axes[i, 1].set_ylabel("Amplitude")
    axes[i, 1].set_ylim(-1, 1)
    axes[i, 1].axhline(0, color="gray", linewidth=0.5)

axes[-1, 0].set_xlabel("Time (s)")
axes[-1, 1].set_xlabel("Time (s)")

plt.suptitle("Waveform Comparison: Clean Speech (EARS) vs. Noise (WHAM!)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Listen to the audio samples
print("=" * 60)
print("EARS (Clean Speech) Samples")
print("=" * 60)
for a in ears_audio:
    print(f"\n{a['filename']} ({a['duration_s']:.1f}s):")
    ipd.display(ipd.Audio(a["waveform"].squeeze().numpy(), rate=a["sample_rate"]))

print("\n" + "=" * 60)
print("WHAM (Noise) Samples")
print("=" * 60)
for a in wham_audio:
    print(f"\n{a['filename']} ({a['duration_s']:.1f}s):")
    # Play only first 5 seconds of potentially long noise files
    max_samples = min(a["waveform"].shape[1], 5 * a["sample_rate"])
    ipd.display(ipd.Audio(a["waveform"][:, :max_samples].squeeze().numpy(), rate=a["sample_rate"]))

In [ ]:
# Amplitude distribution comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, a in enumerate(ears_audio):
    wav = a["waveform"].squeeze().numpy()
    axes[0].hist(wav, bins=200, alpha=0.5, density=True, label=a["filename"][:25])
axes[0].set_title("EARS - Amplitude Distribution", fontweight="bold")
axes[0].set_xlabel("Amplitude")
axes[0].set_ylabel("Density")
axes[0].legend(fontsize=8)
axes[0].set_xlim(-0.5, 0.5)

for i, a in enumerate(wham_audio):
    wav = a["waveform"].squeeze().numpy()
    axes[1].hist(wav, bins=200, alpha=0.5, density=True, label=a["filename"][:25])
axes[1].set_title("WHAM - Amplitude Distribution", fontweight="bold")
axes[1].set_xlabel("Amplitude")
axes[1].set_ylabel("Density")
axes[1].legend(fontsize=8)
axes[1].set_xlim(-0.5, 0.5)

plt.suptitle("Amplitude Histogram Comparison", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Spectrogram Analysis & Comparisons

Compute and visualize different spectrogram representations using torchaudio transforms.

In [ ]:
# Use the project's audio preprocessor config
configs = create_configs()
audio_cfg = configs["audio_preprocessor_cfg"]
norm_cfg = configs["normalizer_cfg"]

print("Audio Preprocessor Config:")
for k, v in vars(audio_cfg).items():
    print(f"  {k}: {v}")
print("\nNormalizer Config:")
for k, v in vars(norm_cfg).items():
    print(f"  {k}: {v}")

In [ ]:
# Pick one EARS and one WHAM sample for detailed spectrogram comparison
clean_wav = ears_audio[0]["waveform"].squeeze()
noise_wav = wham_audio[0]["waveform"].squeeze()
sr = ears_audio[0]["sample_rate"]

# Trim both to 4 seconds
max_len = 4 * sr
clean_wav = clean_wav[:max_len]
noise_wav = noise_wav[:max_len] if noise_wav.shape[0] >= max_len else torch.nn.functional.pad(noise_wav, (0, max_len - noise_wav.shape[0]))

# Create different spectrogram transforms
spec_transform = T.Spectrogram(n_fft=audio_cfg.n_fft, hop_length=audio_cfg.hop_length, power=1)
mel_transform = T.MelSpectrogram(
    sample_rate=sr, n_fft=audio_cfg.n_fft, hop_length=audio_cfg.hop_length,
    n_mels=audio_cfg.n_mels, mel_scale=audio_cfg.mel_scale, norm=audio_cfg.mel_scale
)
mfcc_transform = T.MFCC(sample_rate=sr, n_mfcc=13, melkwargs={"n_fft": audio_cfg.n_fft, "hop_length": audio_cfg.hop_length, "n_mels": audio_cfg.n_mels})
amp_to_db = T.AmplitudeToDB(top_db=audio_cfg.top_db)

# Compute all spectrograms for clean speech
clean_spec = spec_transform(clean_wav)
clean_spec_db = amp_to_db(clean_spec)
clean_mel = mel_transform(clean_wav)
clean_mel_db = amp_to_db(clean_mel)
clean_mfcc = mfcc_transform(clean_wav)

# Compute all spectrograms for noise
noise_spec_db = amp_to_db(spec_transform(noise_wav))
noise_mel_db = amp_to_db(mel_transform(noise_wav))
noise_mfcc = mfcc_transform(noise_wav)

print(f"Spectrogram shape:     {clean_spec_db.shape}")
print(f"Mel spectrogram shape: {clean_mel_db.shape}")
print(f"MFCC shape:            {clean_mfcc.shape}")

In [ ]:
# Multi-representation spectrogram comparison: Clean vs Noise
fig, axes = plt.subplots(3, 2, figsize=(20, 15))

spec_kwargs = {"origin": "lower", "aspect": "auto", "interpolation": "nearest"}

# Row 1: Linear Spectrogram (dB)
im1 = axes[0, 0].imshow(clean_spec_db.numpy(), cmap="magma", **spec_kwargs)
axes[0, 0].set_title("Clean Speech - Spectrogram (dB)", fontweight="bold")
axes[0, 0].set_ylabel("Frequency Bin")
fig.colorbar(im1, ax=axes[0, 0], label="dB")

im2 = axes[0, 1].imshow(noise_spec_db.numpy(), cmap="magma", **spec_kwargs)
axes[0, 1].set_title("Noise - Spectrogram (dB)", fontweight="bold")
fig.colorbar(im2, ax=axes[0, 1], label="dB")

# Row 2: Mel Spectrogram (dB)
im3 = axes[1, 0].imshow(clean_mel_db.numpy(), cmap="inferno", **spec_kwargs)
axes[1, 0].set_title("Clean Speech - Mel Spectrogram (dB)", fontweight="bold")
axes[1, 0].set_ylabel("Mel Bin")
fig.colorbar(im3, ax=axes[1, 0], label="dB")

im4 = axes[1, 1].imshow(noise_mel_db.numpy(), cmap="inferno", **spec_kwargs)
axes[1, 1].set_title("Noise - Mel Spectrogram (dB)", fontweight="bold")
fig.colorbar(im4, ax=axes[1, 1], label="dB")

# Row 3: MFCC
im5 = axes[2, 0].imshow(clean_mfcc.numpy(), cmap="coolwarm", **spec_kwargs)
axes[2, 0].set_title("Clean Speech - MFCC", fontweight="bold")
axes[2, 0].set_ylabel("MFCC Coefficient")
axes[2, 0].set_xlabel("Time Frame")
fig.colorbar(im5, ax=axes[2, 0], label="Value")

im6 = axes[2, 1].imshow(noise_mfcc.numpy(), cmap="coolwarm", **spec_kwargs)
axes[2, 1].set_title("Noise - MFCC", fontweight="bold")
axes[2, 1].set_xlabel("Time Frame")
fig.colorbar(im6, ax=axes[2, 1], label="Value")

plt.suptitle("Spectrogram Representations: Clean Speech vs. Noise",
             fontsize=18, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Spectral analysis: Frequency content comparison
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# Mean spectral energy over time
clean_mean_spec = clean_spec_db.numpy().mean(axis=1)
noise_mean_spec = noise_spec_db.numpy().mean(axis=1)
freqs = np.linspace(0, sr / 2, len(clean_mean_spec))

axes[0, 0].plot(freqs, clean_mean_spec, color="#2ecc71", label="Clean Speech", linewidth=1.5)
axes[0, 0].plot(freqs, noise_mean_spec, color="#e74c3c", label="Noise", linewidth=1.5, alpha=0.8)
axes[0, 0].set_title("Mean Spectral Energy vs Frequency", fontweight="bold")
axes[0, 0].set_xlabel("Frequency (Hz)")
axes[0, 0].set_ylabel("Energy (dB)")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Mean Mel energy over time
clean_mean_mel = clean_mel_db.numpy().mean(axis=1)
noise_mean_mel = noise_mel_db.numpy().mean(axis=1)
mel_bins = np.arange(len(clean_mean_mel))

axes[0, 1].plot(mel_bins, clean_mean_mel, color="#2ecc71", label="Clean Speech", linewidth=1.5)
axes[0, 1].plot(mel_bins, noise_mean_mel, color="#e74c3c", label="Noise", linewidth=1.5, alpha=0.8)
axes[0, 1].set_title("Mean Mel Energy vs Mel Bin", fontweight="bold")
axes[0, 1].set_xlabel("Mel Bin")
axes[0, 1].set_ylabel("Energy (dB)")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Temporal energy (RMS per frame)
clean_temporal = clean_spec_db.numpy().mean(axis=0)
noise_temporal = noise_spec_db.numpy().mean(axis=0)
frames = np.arange(len(clean_temporal))

axes[1, 0].plot(frames, clean_temporal, color="#2ecc71", label="Clean Speech", linewidth=1.0, alpha=0.9)
axes[1, 0].plot(frames, noise_temporal, color="#e74c3c", label="Noise", linewidth=1.0, alpha=0.7)
axes[1, 0].set_title("Mean Energy Over Time (per frame)", fontweight="bold")
axes[1, 0].set_xlabel("Time Frame")
axes[1, 0].set_ylabel("Energy (dB)")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Spectral centroid comparison
_window = torch.hann_window(audio_cfg.n_fft)
clean_centroid = F.spectral_centroid(clean_wav.unsqueeze(0), sample_rate=sr, pad=0,
                                     window=_window, n_fft=audio_cfg.n_fft,
                                     hop_length=audio_cfg.hop_length, win_length=audio_cfg.n_fft).squeeze()
noise_centroid = F.spectral_centroid(noise_wav.unsqueeze(0), sample_rate=sr, pad=0,
                                     window=_window, n_fft=audio_cfg.n_fft,
                                     hop_length=audio_cfg.hop_length, win_length=audio_cfg.n_fft).squeeze()

axes[1, 1].plot(clean_centroid.numpy(), color="#2ecc71", label=f"Clean (mean: {clean_centroid.mean():.0f} Hz)", linewidth=1.0)
axes[1, 1].plot(noise_centroid.numpy(), color="#e74c3c", label=f"Noise (mean: {noise_centroid.mean():.0f} Hz)", linewidth=1.0, alpha=0.7)
axes[1, 1].set_title("Spectral Centroid Over Time", fontweight="bold")
axes[1, 1].set_xlabel("Time Frame")
axes[1, 1].set_ylabel("Frequency (Hz)")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("Spectral Analysis: Clean Speech vs. Noise", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Different window sizes and hop lengths comparison
window_configs = [
    {"n_fft": 256, "hop": 64, "label": "n_fft=256, hop=64"},
    {"n_fft": 512, "hop": 128, "label": "n_fft=512, hop=128"},
    {"n_fft": 1024, "hop": 256, "label": "n_fft=1024, hop=256 (project default)"},
    {"n_fft": 2048, "hop": 512, "label": "n_fft=2048, hop=512"},
]

fig, axes = plt.subplots(2, 2, figsize=(20, 10))

for idx, cfg in enumerate(window_configs):
    ax = axes[idx // 2, idx % 2]
    spec_t = T.Spectrogram(n_fft=cfg["n_fft"], hop_length=cfg["hop"], power=1)
    spec_db = T.AmplitudeToDB(top_db=80)(spec_t(clean_wav))
    im = ax.imshow(spec_db.numpy(), origin="lower", aspect="auto", cmap="magma", interpolation="nearest")
    ax.set_title(cfg["label"], fontsize=12, fontweight="bold")
    ax.set_xlabel("Time Frame")
    ax.set_ylabel("Frequency Bin")
    fig.colorbar(im, ax=ax, label="dB")

plt.suptitle("Effect of Window Size on Spectrogram Resolution (Time-Frequency Tradeoff)",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Clean vs. Noisy Signal Mixing & Visualization

Demonstrate the audio mixing process at different SNR levels using the project's `AudioMixingDataset`.

In [ ]:
# Mix clean speech with noise at different SNR levels
snr_levels = [-5.0, 0.0, 5.0, 10.0, 15.0]

# Trim to 2 seconds for clearer visualization
segment_len = 2 * sr
clean_segment = clean_wav[:segment_len]
noise_segment = noise_wav[:segment_len]

fig = plt.figure(figsize=(22, 18))
gs = gridspec.GridSpec(len(snr_levels) + 2, 3, hspace=0.5, wspace=0.35)

# Row 0: Original clean and noise
time_axis = np.arange(segment_len) / sr

ax_clean_wav = fig.add_subplot(gs[0, 0])
ax_clean_wav.plot(time_axis, clean_segment.numpy(), color="#2ecc71", linewidth=0.5)
ax_clean_wav.set_title("Clean Speech (Waveform)", fontweight="bold", fontsize=11)
ax_clean_wav.set_ylabel("Amplitude")
ax_clean_wav.set_ylim(-1, 1)

ax_clean_spec = fig.add_subplot(gs[0, 1:])
clean_seg_mel_db = amp_to_db(mel_transform(clean_segment))
ax_clean_spec.imshow(clean_seg_mel_db.numpy(), origin="lower", aspect="auto", cmap="magma", interpolation="nearest")
ax_clean_spec.set_title("Clean Speech (Mel Spectrogram dB)", fontweight="bold", fontsize=11)
ax_clean_spec.set_ylabel("Mel Bin")

ax_noise_wav = fig.add_subplot(gs[1, 0])
ax_noise_wav.plot(time_axis, noise_segment.numpy(), color="#e74c3c", linewidth=0.5)
ax_noise_wav.set_title("Noise (Waveform)", fontweight="bold", fontsize=11)
ax_noise_wav.set_ylabel("Amplitude")
ax_noise_wav.set_ylim(-1, 1)

ax_noise_spec = fig.add_subplot(gs[1, 1:])
noise_seg_mel_db = amp_to_db(mel_transform(noise_segment))
ax_noise_spec.imshow(noise_seg_mel_db.numpy(), origin="lower", aspect="auto", cmap="magma", interpolation="nearest")
ax_noise_spec.set_title("Noise (Mel Spectrogram dB)", fontweight="bold", fontsize=11)
ax_noise_spec.set_ylabel("Mel Bin")

# Rows 2+: Mixed at different SNR levels
for i, snr in enumerate(snr_levels):
    snr_tensor = torch.tensor([snr])
    mixed = F.add_noise(clean_segment.unsqueeze(0), noise_segment.unsqueeze(0), snr_tensor).squeeze()

    ax_wav = fig.add_subplot(gs[i + 2, 0])
    ax_wav.plot(time_axis, mixed.numpy(), color="#3498db", linewidth=0.5)
    ax_wav.set_title(f"Mixed @ SNR={snr:.0f} dB (Waveform)", fontweight="bold", fontsize=11)
    ax_wav.set_ylabel("Amplitude")
    ax_wav.set_ylim(-1, 1)

    ax_spec = fig.add_subplot(gs[i + 2, 1:])
    mixed_mel_db = amp_to_db(mel_transform(mixed))
    ax_spec.imshow(mixed_mel_db.numpy(), origin="lower", aspect="auto", cmap="magma", interpolation="nearest")
    ax_spec.set_title(f"Mixed @ SNR={snr:.0f} dB (Mel Spectrogram dB)", fontweight="bold", fontsize=11)
    ax_spec.set_ylabel("Mel Bin")

    if i == len(snr_levels) - 1:
        ax_wav.set_xlabel("Time (s)")
        ax_spec.set_xlabel("Time Frame")

plt.suptitle("Signal Mixing at Different SNR Levels", fontsize=18, fontweight="bold", y=1.01)
plt.show()

In [ ]:
# Listen to mixed audio at different SNR levels
print("Audio Playback at Different SNR Levels")
print("=" * 50)

print("\nOriginal Clean Speech:")
ipd.display(ipd.Audio(clean_segment.numpy(), rate=sr))

print("\nOriginal Noise:")
ipd.display(ipd.Audio(noise_segment.numpy(), rate=sr))

for snr in snr_levels:
    snr_tensor = torch.tensor([snr])
    mixed = F.add_noise(clean_segment.unsqueeze(0), noise_segment.unsqueeze(0), snr_tensor).squeeze()
    mixed = torch.clamp(mixed, -1.0, 1.0)
    print(f"\nMixed @ SNR = {snr:.0f} dB:")
    ipd.display(ipd.Audio(mixed.numpy(), rate=sr))

In [ ]:
# SNR effect on spectral difference
fig, axes = plt.subplots(1, len(snr_levels), figsize=(5 * len(snr_levels), 5))

clean_seg_mel_db_np = clean_seg_mel_db.numpy()

for i, snr in enumerate(snr_levels):
    snr_tensor = torch.tensor([snr])
    mixed = F.add_noise(clean_segment.unsqueeze(0), noise_segment.unsqueeze(0), snr_tensor).squeeze()
    mixed_mel_db = amp_to_db(mel_transform(mixed)).numpy()
    
    diff = mixed_mel_db - clean_seg_mel_db_np
    
    im = axes[i].imshow(diff, origin="lower", aspect="auto", cmap="RdBu_r",
                        vmin=-30, vmax=30, interpolation="nearest")
    axes[i].set_title(f"SNR={snr:.0f} dB\nΔ = Mixed - Clean", fontweight="bold", fontsize=11)
    axes[i].set_xlabel("Time Frame")
    if i == 0:
        axes[i].set_ylabel("Mel Bin")
    fig.colorbar(im, ax=axes[i], label="dB Difference", shrink=0.8)

plt.suptitle("Spectral Difference (Mixed - Clean) at Different SNR Levels",
             fontsize=16, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

---
## 7. Data Pipeline Visualization

Visualize the full data processing pipeline used by the `mel_bin2bin` model: Raw Audio → Mel Spectrogram → dB Scale → Normalization → Trimming.

In [ ]:
# Instantiate individual pipeline stages from the project
mel_processor = MelSpectrogramProcessor(config=audio_cfg)
db_scaler = AudioToDBScaler(audio_cfg)
normalizer = MinMaxFixedNormalizer(norm_cfg)

# Mix a sample at moderate SNR
snr_tensor = torch.tensor([5.0])
mixed_signal = F.add_noise(clean_segment.unsqueeze(0), noise_segment.unsqueeze(0), snr_tensor).squeeze()

# Step through pipeline stages
stage_1_mel_clean = mel_processor(clean_segment)       # [1, n_mels, time]
stage_1_mel_noisy = mel_processor(mixed_signal)

stage_2_db_clean = db_scaler(stage_1_mel_clean)         # [1, n_mels, time] in dB
stage_2_db_noisy = db_scaler(stage_1_mel_noisy)

stage_3_norm_clean = normalizer(stage_2_db_clean)       # [1, n_mels, time] normalized
stage_3_norm_noisy = normalizer(stage_2_db_noisy)

# Trim to project's target shape
target_shape = audio_cfg.max_spec_shapes
stage_4_trim_clean = stage_3_norm_clean[:, :target_shape[0], :target_shape[1]]
stage_4_trim_noisy = stage_3_norm_noisy[:, :target_shape[0], :target_shape[1]]

print("Pipeline Stage Shapes:")
print(f"  Raw audio:        {clean_segment.shape}")
print(f"  Mel spectrogram:  {stage_1_mel_clean.shape}")
print(f"  After dB scale:   {stage_2_db_clean.shape}")
print(f"  After normalize:  {stage_3_norm_clean.shape}")
print(f"  After trim:       {stage_4_trim_clean.shape} (target: {target_shape})")

In [ ]:
# Full pipeline visualization for both clean and noisy signals
stages = [
    ("1. Mel Spectrogram\n(Amplitude)", stage_1_mel_clean, stage_1_mel_noisy, "viridis", None),
    ("2. dB Scale", stage_2_db_clean, stage_2_db_noisy, "magma", None),
    ("3. Normalized\n(MinMax → [-1, 1])", stage_3_norm_clean, stage_3_norm_noisy, "inferno", (-1, 1)),
    ("4. Trimmed\n(80×128)", stage_4_trim_clean, stage_4_trim_noisy, "magma", (-1, 1)),
]

fig, axes = plt.subplots(len(stages), 2, figsize=(16, 4 * len(stages)))

for row, (title, clean_s, noisy_s, cmap, vlim) in enumerate(stages):
    clean_np = clean_s.squeeze().detach().numpy()
    noisy_np = noisy_s.squeeze().detach().numpy()
    
    kwargs = {"origin": "lower", "aspect": "auto", "cmap": cmap, "interpolation": "nearest"}
    if vlim:
        kwargs["vmin"], kwargs["vmax"] = vlim

    im1 = axes[row, 0].imshow(clean_np, **kwargs)
    axes[row, 0].set_title(f"{title} — Clean", fontweight="bold", fontsize=11)
    axes[row, 0].set_ylabel("Mel Bin")
    fig.colorbar(im1, ax=axes[row, 0], shrink=0.8)

    im2 = axes[row, 1].imshow(noisy_np, **kwargs)
    axes[row, 1].set_title(f"{title} — Noisy (SNR=5 dB)", fontweight="bold", fontsize=11)
    fig.colorbar(im2, ax=axes[row, 1], shrink=0.8)

axes[-1, 0].set_xlabel("Time Frame")
axes[-1, 1].set_xlabel("Time Frame")

plt.suptitle("mel_bin2bin Data Pipeline: Stage-by-Stage Visualization",
             fontsize=18, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Value distribution at each pipeline stage
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

stage_data = [
    ("Mel Spectrogram (Amplitude)", stage_1_mel_clean, stage_1_mel_noisy),
    ("After dB Conversion", stage_2_db_clean, stage_2_db_noisy),
    ("After Normalization [-1, 1]", stage_3_norm_clean, stage_3_norm_noisy),
    ("After Trimming (80×128)", stage_4_trim_clean, stage_4_trim_noisy),
]

for idx, (title, clean_s, noisy_s) in enumerate(stage_data):
    ax = axes[idx // 2, idx % 2]
    c_flat = clean_s.flatten().detach().numpy()
    n_flat = noisy_s.flatten().detach().numpy()
    
    ax.hist(c_flat, bins=100, alpha=0.6, color="#2ecc71", density=True, label="Clean")
    ax.hist(n_flat, bins=100, alpha=0.6, color="#e74c3c", density=True, label="Noisy")
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.set_xlabel("Value")
    ax.set_ylabel("Density")
    ax.legend()
    ax.grid(True, alpha=0.3)

    stats_text = (f"Clean: [{c_flat.min():.2f}, {c_flat.max():.2f}], μ={c_flat.mean():.2f}\n"
                  f"Noisy: [{n_flat.min():.2f}, {n_flat.max():.2f}], μ={n_flat.mean():.2f}")
    ax.text(0.02, 0.95, stats_text, transform=ax.transAxes, fontsize=8,
            verticalalignment="top", bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.suptitle("Value Distributions at Each Pipeline Stage", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Train/Val/Test split statistics
splits = split_dataframes(ears_df, wham_df, train_percentage=0.8, reduced_size=None)

split_stats = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "EARS Samples": [len(splits["train_ears_df"]), len(splits["val_ears_df"]), len(splits["test_ears_df"])],
    "WHAM Samples": [len(splits["train_wham_df"]), len(splits["val_wham_df"]), len(splits["test_wham_df"])],
})
split_stats["EARS %"] = (split_stats["EARS Samples"] / split_stats["EARS Samples"].sum() * 100).round(1)
split_stats["WHAM %"] = (split_stats["WHAM Samples"] / split_stats["WHAM Samples"].sum() * 100).round(1)

print("Dataset Split Summary:")
print(split_stats.to_string(index=False))

# Visualize split proportions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_split = ["#3498db", "#f39c12", "#e74c3c"]

axes[0].bar(split_stats["Split"], split_stats["EARS Samples"], color=colors_split, edgecolor="black")
axes[0].set_title("EARS Dataset Splits", fontweight="bold")
axes[0].set_ylabel("Number of Samples")
for i, v in enumerate(split_stats["EARS Samples"]):
    axes[0].text(i, v + 20, f"{v}\n({split_stats['EARS %'].iloc[i]}%)", ha="center", fontweight="bold")

axes[1].bar(split_stats["Split"], split_stats["WHAM Samples"], color=colors_split, edgecolor="black")
axes[1].set_title("WHAM Dataset Splits", fontweight="bold")
axes[1].set_ylabel("Number of Samples")
for i, v in enumerate(split_stats["WHAM Samples"]):
    axes[1].text(i, v + 20, f"{v}\n({split_stats['WHAM %'].iloc[i]}%)", ha="center", fontweight="bold")

plt.suptitle("Train / Validation / Test Split Proportions", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Training config summary
train_cfg = configs["train_cfg"]
mixing_cfg = configs["mixing_audio_cfg"]

config_summary = pd.DataFrame([
    {"Category": "Mixing", "Parameter": k, "Value": str(v)} for k, v in vars(mixing_cfg).items()
] + [
    {"Category": "Audio", "Parameter": k, "Value": str(v)} for k, v in vars(audio_cfg).items()
] + [
    {"Category": "Normalizer", "Parameter": k, "Value": str(v)} for k, v in vars(norm_cfg).items()
] + [
    {"Category": "Training", "Parameter": k, "Value": str(v)} for k, v in vars(train_cfg).items()
])

print("\nComplete Configuration Summary:")
print("=" * 60)
for category in config_summary["Category"].unique():
    print(f"\n[{category}]")
    subset = config_summary[config_summary["Category"] == category]
    for _, row in subset.iterrows():
        print(f"  {row['Parameter']:30s} = {row['Value']}")